# 07 · From lift to money: iROAS and profit

**Question:** the campaign clearly worked (+6.9% sessions, p < 0.001). Was it worth the money?

**Answer:** no. It drove ~716 incremental sessions and ~$1,066 of incremental revenue on $800 of spend: **iROAS 1.33** (95% range 0.57–1.97). With a 50% gross margin, break-even needs iROAS ≥ **2.0**, so the campaign **loses money in every scenario**. The platform's reported ROAS of 3.5 overstates the return about 2.6×.

**Setup:** the 6-state test from [04](04_placebo_inference.ipynb) (true lift 8%, DiD estimate 6.9%). Spend ($800), margin (50%), and platform-attributed revenue ($2,800) are assumptions to illustrate the logic; the public sample is scaled down, so the ratios matter, not the dollar amounts.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("data/ga4_state_daily.csv", parse_dates=["date"])
START = pd.Timestamp("2021-01-04")
WINDOW = pd.Timedelta(days=28)

wide = df.pivot(index="date", columns="state", values="sessions").fillna(0).astype(float)
daily_median = df.groupby("state").sessions.median()
wide = wide[daily_median[daily_median >= 10].index]
pre  = (wide.index >= START - WINDOW) & (wide.index < START)
post = (wide.index >= START) & (wide.index < START + WINDOW)

def did_lift(data, treated, control):
    T = data[treated].sum(axis=1); C = data[control].sum(axis=1)
    return (T[post].sum() / T[pre].sum()) / (C[post].sum() / C[pre].sum()) - 1

TREATED = ["Texas", "Florida", "Georgia", "Arizona", "Ohio", "Michigan"]
CONTROL = [s for s in wide.columns if s not in TREATED]

observed = wide.copy()
observed.loc[post, TREATED] *= 1.08
lift = did_lift(observed, TREATED, CONTROL)

rng = np.random.default_rng(42)
placebos = []
for _ in range(1000):
    fake = list(rng.choice(CONTROL, 6, replace=False))
    placebos.append(did_lift(wide, fake, [s for s in CONTROL if s not in fake]))
placebos = np.array(placebos)
lift_low  = lift - np.quantile(placebos, 0.975)
lift_high = lift - np.quantile(placebos, 0.025)

SPEND = 800
print(f"Lift {lift:.1%}, 95% CI [{lift_low:.1%}, {lift_high:.1%}] | Spend ${SPEND}")

Lift 6.9%, 95% CI [2.8%, 10.5%] | Spend $800


## Step 1 · % lift → incremental sessions

Lift is relative to the **counterfactual**: actual = counterfactual × (1 + lift). The counterfactual is the *treated* states without ads, not the control states' total.

In [2]:
actual = observed.loc[post, TREATED].sum().sum()     # total sessions in treated states during the test
counterfactual = actual / (1 + lift)
incremental_sessions = actual - counterfactual

print(f"Actual: {actual:,.0f} | Counterfactual: {counterfactual:,.0f} | Incremental: {incremental_sessions:,.0f}")
print(f"(Shortcut actual × lift = {actual * lift:,.0f}: overstates, because it applies the lift to a base that already includes it)")

Actual: 11,076 | Counterfactual: 10,361 | Incremental: 716
(Shortcut actual × lift = 765: overstates, because it applies the lift to a base that already includes it)


## Step 2 · Incremental sessions → incremental revenue

Daily revenue per state is far noisier than sessions (most state-days have no purchase), so revenue is modeled as incremental sessions × CVR × AOV, using the treated states' pre-period funnel.

In [3]:
# How noisy is each KPI? (coefficient of variation = std / mean of daily values, median across states)
cv = lambda col: (df.pivot(index="date", columns="state", values=col)[wide.columns].pipe(lambda x: x.std() / x.mean())).median()
print(f"Daily noise (CV): sessions {cv('sessions'):.2f} | revenue {cv('revenue'):.2f}")

hist_tr = df[df.state.isin(TREATED) & (df.date < START)]
cvr = hist_tr["purchases"].sum() / hist_tr["sessions"].sum()     # purchases / sessions in hist_tr
aov = hist_tr["revenue"].sum() / hist_tr["purchases"].sum()     # revenue / purchases in hist_tr
incremental_revenue = incremental_sessions * cvr * aov
print(f"CVR {cvr:.2%} | AOV ${aov:.2f} | Incremental revenue ${incremental_revenue:,.0f}")

Daily noise (CV): sessions 0.33 | revenue 2.44
CVR 2.01% | AOV $73.95 | Incremental revenue $1,066


## Step 3 · iROAS

In [4]:
iroas = incremental_revenue / SPEND
cpis  = SPEND / incremental_sessions
print(f"iROAS: {iroas:.2f} | Cost per incremental session: ${cpis:.2f}")

iROAS: 1.33 | Cost per incremental session: $1.12


## Step 4 · Uncertainty and profit (50% gross margin)

In [5]:
MARGIN = 0.50

def iroas_from_lift(l):
    inc_sessions = actual - actual / (1 + l)
    return inc_sessions * cvr * aov / SPEND

rows = []
for name, l in [("CI low", lift_low), ("estimate", lift), ("CI high", lift_high)]:
    r = iroas_from_lift(l)
    profit = r * SPEND * MARGIN - SPEND     # incremental revenue × margin − spend
    rows.append({"case": name, "lift": l, "iROAS": r, "profit": profit})
pd.DataFrame(rows).style.format({"lift": "{:.1%}", "iROAS": "{:.2f}", "profit": "${:,.0f}"})

,case,lift,iROAS,profit
0,CI low,2.8%,0.57,$-573
1,estimate,6.9%,1.33,$-267
2,CI high,10.5%,1.97,$-13


In [6]:
print(f"Profit break-even iROAS = 1 / {MARGIN} = {1 / MARGIN:.1f}")

Profit break-even iROAS = 1 / 0.5 = 2.0


## Step 5 · Platform ROAS vs iROAS

In [7]:
PLATFORM_REVENUE = 2800
platform_roas = PLATFORM_REVENUE / SPEND
share_incremental = incremental_revenue / PLATFORM_REVENUE
print(f"Platform ROAS {platform_roas:.1f} vs iROAS {iroas:.2f}")
print(f"Only {share_incremental:.0%} of platform-attributed revenue was truly incremental")

Platform ROAS 3.5 vs iROAS 1.33
Only 38% of platform-attributed revenue was truly incremental


## Findings

| | Value |
|---|---|
| Actual / counterfactual sessions | 11,076 / 10,361 |
| **Incremental sessions** | **716** (shortcut actual × lift: 765) |
| CVR / AOV (pre-period) | 2.01% / $73.95 |
| Daily noise (CV): sessions vs revenue | 0.33 vs 2.44 |
| **Incremental revenue** | **≈ $1,066** |
| **iROAS** | **1.33** |
| Cost per incremental session | $1.12 |

| Case | Lift | iROAS | Profit |
|---|---|---|---|
| CI low | 2.8% | 0.57 | −$574 |
| Estimate | 6.9% | 1.33 | −$267 |
| CI high | 10.5% | 1.97 | −$13 |

- **Revenue view:** the iROAS range includes 1.0, so it's unclear whether the campaign even pays back in revenue.
- **Profit view:** break-even iROAS = 1 / margin = **2.0**; even the optimistic case falls short.
- **Attribution vs incrementality:** the platform credits $2,800 (ROAS 3.5); only **~38%** of it was incremental. Attribution counts customers who would have bought anyway, retargeted users already about to buy, and conversions other platforms also claim.

## Recommendation

**Don't scale. Pause or reduce the campaign and re-test a cheaper version.**

1. **Test a lower spend level**: returns usually diminish with spend, so the first dollars may be profitable even if the last ones aren't.
2. **Improve efficiency**: target higher-margin products or new customers rather than retargeting likely buyers.
3. **Budget on incrementality, not platform ROAS**: apply the ~0.38 incrementality factor between tests and use the result to calibrate the MMM.

**Caveats:** incremental visitors probably convert worse than the baseline CVR, so iROAS is likely optimistic; the 4-week window misses long-term effects; at a 70% margin, break-even would be 1.43 and the estimate would sit near it.